# Gold Layer - Fraud Detection with XGBoost

This notebook builds the Gold layer for the fraud detection project. It loads the cleaned Silver parquet files, joins supporting datasets, engineers fraud-related features, trains an XGBoost model, and saves model and analytical outputs.

### Notebook Flow

1. Resolve project paths and validate required Silver inputs.
2. Load and standardise Silver datasets.
3. Join source tables into a Gold modelling dataset.
4. Run quality checks and engineer model features.
5. Train, evaluate, and save model outputs.

In [3]:
# =========================================================================
# Import libraries to prepare for data processing and model training.
# =========================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    )

from xgboost import XGBClassifier

In [4]:
# =========================================================================
# Resolve project folders using stable path checks for common run locations.
# =========================================================================

current_dir = Path.cwd().resolve()
candidate_data_dirs = [
    current_dir / "Data",
    current_dir.parent / "Data",
]
data_dir = next((path for path in candidate_data_dirs if path.exists()), candidate_data_dirs[0])
if not data_dir.exists():
    checked = "\n - ".join(str(path) for path in candidate_data_dirs)
    raise FileNotFoundError(f"Data directory not found. Checked:\n - {checked}")

silver_dir = data_dir / "silver"
gold_dir = data_dir / "gold"

if not silver_dir.exists():
    raise FileNotFoundError(f"Silver directory not found: {silver_dir}")

gold_dir.mkdir(parents=True, exist_ok=True)

print("Current working directory:", current_dir)
print("Data directory:", data_dir)
print("Silver directory:", silver_dir)
print("Gold directory:", gold_dir)

Current working directory: C:\Users\9370892\OneDrive - Lloyds Banking Group\Apprenticeship - Data Science\Data Science Pro Practice\Assessment\Scripts
Data directory: C:\Users\9370892\OneDrive - Lloyds Banking Group\Apprenticeship - Data Science\Data Science Pro Practice\Assessment\Data
Silver directory: C:\Users\9370892\OneDrive - Lloyds Banking Group\Apprenticeship - Data Science\Data Science Pro Practice\Assessment\Data\silver
Gold directory: C:\Users\9370892\OneDrive - Lloyds Banking Group\Apprenticeship - Data Science\Data Science Pro Practice\Assessment\Data\gold


In [5]:
# =========================================================================
# Validate and load cleaned Silver datasets stored in parquet format.
# =========================================================================

silver_files = {
    "transactions": silver_dir / "silver_transactions.parquet",
    "users": silver_dir / "silver_users.parquet",
    "cards": silver_dir / "silver_cards.parquet",
    "fraud_labels": silver_dir / "silver_fraud_labels.parquet",
    "mcc_codes": silver_dir / "silver_mcc_codes.parquet",
}

missing_silver_files = [
    str(path)
    for path in silver_files.values()
    if not path.exists()
]
if missing_silver_files:
    missing_text = "\n - ".join(missing_silver_files)
    raise FileNotFoundError(
        f"Required Silver files are missing:\n - {missing_text}"
    )

transactions = pd.read_parquet(silver_files["transactions"])
users = pd.read_parquet(silver_files["users"])
cards = pd.read_parquet(silver_files["cards"])
fraud_labels = pd.read_parquet(silver_files["fraud_labels"])
mcc_df = pd.read_parquet(silver_files["mcc_codes"])

print("Silver files loaded successfully.")
print("Transactions shape:", transactions.shape)
print("Users shape:", users.shape)
print("Cards shape:", cards.shape)
print("Fraud labels shape:", fraud_labels.shape)
print("MCC codes shape:", mcc_df.shape)

Silver files loaded successfully.
Transactions shape: (13305915, 16)
Users shape: (2000, 19)
Cards shape: (6146, 13)
Fraud labels shape: (8914963, 3)
MCC codes shape: (109, 2)


In [6]:
# =========================================================================
# Standardise column names for consistent joins and references.
# =========================================================================

datasets = {
    "transactions": transactions,
    "users": users,
    "cards": cards,
    "fraud_labels": fraud_labels,
    "mcc_codes": mcc_df,
}

for dataframe in datasets.values():
    dataframe.columns = dataframe.columns.str.lower().str.strip()

for name, dataframe in datasets.items():
    print(f"{name} columns:")
    print(dataframe.columns.tolist())
    print()

transactions columns:
['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'snapshot_date', 'ingestion_timestamp', 'is_refund', 'amount_outlier_flag']

users columns:
['id', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'snapshot_date', 'ingestion_timestamp', 'per_capita_income_outlier_flag', 'yearly_income_outlier_flag', 'total_debt_outlier_flag', 'credit_score_outlier_flag']

cards columns:
['id', 'client_id', 'card_brand', 'card_type', 'expires', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web', 'snapshot_date', 'ingestion_timestamp']

fraud_labels columns:
['transaction_id', 'fraud_label', 'is_fraud']

mcc_codes columns:
['mcc', 'merchant_category']



In [7]:
# =========================================================================
# Joining tables together, adding fraud labels to the transactions dataset.
# =========================================================================

gold_df = transactions.merge(
    fraud_labels,
    left_on="id",
    right_on="transaction_id",
    how="left"
)

print("Gold shape after fraud label join:", gold_df.shape)
print("Fraud target distribution after join:")
print(gold_df["is_fraud"].value_counts(dropna=False))

Gold shape after fraud label join: (13305915, 19)
Fraud target distribution after join:
is_fraud
0.0    8901631
NaN    4390952
1.0      13332
Name: count, dtype: int64


In [8]:
# =========================================================================
# Continue joining the gold dataset with the users dataset to add user detail.
# =========================================================================

gold_df = gold_df.merge(
    users,
    left_on="client_id",
    right_on="id",
    how="left",
    suffixes=("", "_user")
)

print("Gold shape after user join:", gold_df.shape)

Gold shape after user join: (13305915, 38)


In [9]:
# =========================================================================
# Enrich transactions with card-level attributes.
# =========================================================================

gold_df = gold_df.merge(
    cards,
    left_on="card_id",
    right_on="id",
    how="left",
    suffixes=("", "_card")
)

print("Gold shape after card join:", gold_df.shape)

Gold shape after card join: (13305915, 51)


In [10]:
# =========================================================================
# Add merchant category context using the MCC reference table.
# =========================================================================

gold_df = gold_df.merge(
    mcc_df,
    on="mcc",
    how="left"
)

print("Gold shape after MCC join:", gold_df.shape)
print("MCC category sample:")
display(gold_df[["mcc", "merchant_category"]].head(10))

print("Missing merchant category count:",
      gold_df["merchant_category"].isna().sum())

Gold shape after MCC join: (13305915, 52)
MCC category sample:


,mcc,merchant_category
0,5499,Miscellaneous Food Stores
1,5311,Department Stores
2,4829,Money Transfer
3,4829,Money Transfer
4,5813,Drinking Places (Alcoholic Beverages)
5,5942,Book Stores
6,5499,Miscellaneous Food Stores
7,4784,Tolls and Bridge Fees
8,7801,"Athletic Fields, Commercial Sports"
9,5813,Drinking Places (Alcoholic Beverages)


Missing merchant category count: 0


In [11]:
# =========================================================================
# Final overview of the gold dataset.
# =========================================================================

print("Gold dataset shape:", gold_df.shape)
print("Gold columns:")
print(gold_df.columns.tolist())

display(gold_df.head())

Gold dataset shape: (13305915, 52)
Gold columns:
['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'snapshot_date', 'ingestion_timestamp', 'is_refund', 'amount_outlier_flag', 'transaction_id', 'fraud_label', 'is_fraud', 'id_user', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'snapshot_date_user', 'ingestion_timestamp_user', 'per_capita_income_outlier_flag', 'yearly_income_outlier_flag', 'total_debt_outlier_flag', 'credit_score_outlier_flag', 'id_card', 'client_id_card', 'card_brand', 'card_type', 'expires', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web', 'snapshot_date_card', 'ingestion_timestamp_card', 'merchant_category']


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,expires,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,snapshot_date_card,ingestion_timestamp_card,merchant_category
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,...,2022-07-01,True,2,55.0,2008-05-01,2008,False,20260824,2026-08-24T14:27:39,Miscellaneous Food Stores
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,...,2024-12-01,True,1,9100.0,2005-09-01,2015,False,20260824,2026-08-24T14:27:39,Department Stores
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084.0,...,2020-05-01,True,1,14802.0,2006-01-01,2008,False,20260824,2026-08-24T14:27:39,Money Transfer
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,...,2024-10-01,False,2,37634.0,2004-05-01,2006,False,20260824,2026-08-24T14:27:39,Money Transfer
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776.0,...,2020-01-01,True,1,19113.0,2009-07-01,2014,False,20260824,2026-08-24T14:27:39,Drinking Places (Alcoholic Beverages)


In [12]:
# =========================================================================
# Quality Checks: Review missing values after the joins have been completed.
# =========================================================================

missing_summary = gold_df.isna().sum().sort_values(ascending=False)

display(missing_summary.head(40))

print("Fraud target distribution:")
display(gold_df["is_fraud"].value_counts(dropna=False))

print("Fraud target percentage:")
display(gold_df["is_fraud"].value_counts(normalize=True, dropna=False) * 100)

errors                            13094522
fraud_label                        4390952
transaction_id                     4390952
is_fraud                           4390952
zip                                1652706
merchant_state                     1563700
date                                     0
id                                       0
card_id                                  0
client_id                                0
merchant_city                            0
amount                                   0
mcc                                      0
use_chip                                 0
merchant_id                              0
snapshot_date                            0
amount_outlier_flag                      0
is_refund                                0
ingestion_timestamp                      0
id_user                                  0
current_age                              0
retirement_age                           0
birth_year                               0
birth_month

Fraud target distribution:


is_fraud
0.0    8901631
NaN    4390952
1.0      13332
Name: count, dtype: int64

Fraud target percentage:


is_fraud
0.0    66.899804
NaN    33.000000
1.0     0.100196
Name: proportion, dtype: float64

In [13]:
# =========================================================================
# Convert date-like columns to datetime for date-based features.
# =========================================================================

date_columns = [
    "date",
    "snapshot_date",
    "snapshot_date_user",
    "snapshot_date_card",
    "ingestion_timestamp",
    "ingestion_timestamp_user",
    "ingestion_timestamp_card",
    "expires",
    "acct_open_date",
]

for column in date_columns:
    if column in gold_df.columns:
        gold_df[column] = pd.to_datetime(gold_df[column], errors="coerce")

print("Date conversion complete.")

Date conversion complete.


In [14]:
# =========================================================================
# Create time-based features from the transaction date.
# =========================================================================

if "date" in gold_df.columns:
    gold_df["transaction_hour"] = gold_df["date"].dt.hour
    gold_df["transaction_day"] = gold_df["date"].dt.day
    gold_df["transaction_month"] = gold_df["date"].dt.month
    gold_df["transaction_dayofweek"] = gold_df["date"].dt.dayofweek
    gold_df["is_weekend"] = gold_df["transaction_dayofweek"].isin([
                                                                  5, 6]).astype(int)

# Create a customer affordability/risk ratio.
if "total_debt" in gold_df.columns and "yearly_income" in gold_df.columns:
    gold_df["debt_to_income_ratio"] = (
        gold_df["total_debt"] / gold_df["yearly_income"].replace(0, np.nan)
    )

# Compare income against available credit limit.
if "yearly_income" in gold_df.columns and "credit_limit" in gold_df.columns:
    gold_df["income_to_credit_ratio"] = (
        gold_df["yearly_income"] / gold_df["credit_limit"].replace(0, np.nan)
    )

# Create simple credit score risk indicators.
if "credit_score" in gold_df.columns:
    gold_df["low_credit_score_flag"] = (
        gold_df["credit_score"] < 600).astype(int)
    gold_df["high_credit_score_flag"] = (
        gold_df["credit_score"] >= 750).astype(int)

# Flag customers with debt greater than annual income.
if "debt_to_income_ratio" in gold_df.columns:
    gold_df["high_debt_ratio_flag"] = (
        gold_df["debt_to_income_ratio"] > 1).astype(int)

# Flag unusually high transaction amounts using percentile thresholds.
if "amount" in gold_df.columns:
    high_amount_threshold = gold_df["amount"].quantile(0.95)
    very_high_amount_threshold = gold_df["amount"].quantile(0.99)

    gold_df["high_amount_flag"] = (
        gold_df["amount"] > high_amount_threshold).astype(int)
    gold_df["very_high_amount_flag"] = (
        gold_df["amount"] > very_high_amount_threshold).astype(int)

    print("High amount threshold:", high_amount_threshold)
    print("Very high amount threshold:", very_high_amount_threshold)

# Convert card-on-dark-web values into a numeric risk flag.
if "card_on_dark_web" in gold_df.columns:
    gold_df["card_on_dark_web_flag"] = (
        gold_df["card_on_dark_web"]
        .astype(str)
        .str.lower()
        .isin(["yes", "true", "1"])
    ).astype(int)

# Convert has-chip values into a numeric flag.
if "has_chip" in gold_df.columns:
    gold_df["has_chip_flag"] = (
        gold_df["has_chip"]
        .astype(str)
        .str.lower()
        .isin(["yes", "true", "1"])
    ).astype(int)

# Create a binary flag for transactions containing an error value.
if "errors" in gold_df.columns:
    gold_df["has_transaction_error"] = gold_df["errors"].notna().astype(int)

# Ensure the refund indicator is numeric.
if "is_refund" in gold_df.columns:
    gold_df["is_refund"] = gold_df["is_refund"].astype(int)

print("Feature engineering complete.")

High amount threshold: 145.92
Very high amount threshold: 315.95
Feature engineering complete.


## 13. Define the Target Column

The `is_fraud` field is used as the supervised model target.


In [15]:
# =========================================================================
# Define the model target column and clean the gold dataset for modeling.
# =========================================================================

target_col = "is_fraud"

# Remove rows where the model target is missing.
gold_df = gold_df.dropna(subset=[target_col]).copy()
gold_df[target_col] = gold_df[target_col].astype(int)

print("Target column:", target_col)
print("Target distribution:")
print(gold_df[target_col].value_counts())

Target column: is_fraud
Target distribution:
is_fraud
0    8901631
1      13332
Name: count, dtype: int64


In [16]:
# =========================================================================
# Preparing the model setup by removing identifiers, raw dates,
# duplicated join fields or target leakage columns.
# =========================================================================

model_df = gold_df.copy()

# Remove fields that are identifiers, raw dates, duplicated join fields or
# target leakage columns.
drop_cols = [
    target_col,
    "fraud_label",
    "transaction_id",
    "id",
    "id_user",
    "id_card",
    "client_id",
    "card_id",
    "merchant_city",
    "merchant_state",
    "zip",
    "snapshot_date",
    "snapshot_date_user",
    "snapshot_date_card",
    "ingestion_timestamp",
    "ingestion_timestamp_user",
    "ingestion_timestamp_card",
    "date",
    "expires",
    "acct_open_date"
]

# Remove any remaining datetime columns automatically.
remaining_datetime_cols = model_df.select_dtypes(
    include=["datetime64[ns]", "datetimetz"]
).columns.tolist()

drop_cols = list(set(drop_cols + remaining_datetime_cols))

X = model_df.drop(columns=drop_cols, errors="ignore")
y = model_df[target_col]

print("Initial feature shape:", X.shape)
print("Target shape:", y.shape)
print("Columns before encoding:")
print(X.columns.tolist())

Initial feature shape: (8914963, 47)
Target shape: (8914963,)
Columns before encoding:
['amount', 'use_chip', 'merchant_id', 'mcc', 'errors', 'is_refund', 'amount_outlier_flag', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'per_capita_income_outlier_flag', 'yearly_income_outlier_flag', 'total_debt_outlier_flag', 'credit_score_outlier_flag', 'client_id_card', 'card_brand', 'card_type', 'has_chip', 'num_cards_issued', 'credit_limit', 'year_pin_last_changed', 'card_on_dark_web', 'merchant_category', 'transaction_hour', 'transaction_day', 'transaction_month', 'transaction_dayofweek', 'is_weekend', 'debt_to_income_ratio', 'income_to_credit_ratio', 'low_credit_score_flag', 'high_credit_score_flag', 'high_debt_ratio_flag', 'high_amount_flag', 'very_high_amount_flag', 'card_on_dark_web_flag', 'has_chip_flag', 'has_transaction_error']


In [17]:
# =========================================================================
# Converting text fields such as merchant category, card type, and user country into numeric variables.
# =========================================================================

# Identify categorical columns that need encoding.
categorical_cols = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Categorical columns to encode:")
print(categorical_cols)

# Convert categorical fields into numeric dummy variables.
X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

# Replace invalid numeric values and fill remaining missing data.
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

print("Final feature shape after encoding:", X.shape)
display(X.head())

Categorical columns to encode:
['use_chip', 'errors', 'amount_outlier_flag', 'gender', 'per_capita_income_outlier_flag', 'yearly_income_outlier_flag', 'total_debt_outlier_flag', 'credit_score_outlier_flag', 'card_brand', 'card_type', 'has_chip', 'card_on_dark_web', 'merchant_category']


C:\Users\9370892\AppData\Local\Temp\ipykernel_49300\4198560467.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


Final feature shape after encoding: (8914963, 176)


,amount,merchant_id,mcc,is_refund,current_age,retirement_age,birth_year,birth_month,latitude,longitude,...,merchant_category_Theatrical Producers,merchant_category_Tolls and Bridge Fees,"merchant_category_Tools, Parts, Supplies Manufacturing",merchant_category_Towing Services,merchant_category_Travel Agencies,merchant_category_Upholstery and Drapery Stores,"merchant_category_Utilities - Electric, Gas, Water, Sanitary",merchant_category_Welding Repair,merchant_category_Wholesale Clubs,merchant_category_Women's Ready-To-Wear Stores
0,-77.00,59935,5499,1,30,67,1989,7,46.80,-100.76,...,False,False,False,False,False,False,False,False,False,False
1,14.57,67570,5311,0,48,67,1971,6,40.80,-91.12,...,False,False,False,False,False,False,False,False,False,False
2,80.00,27092,4829,0,49,65,1970,4,33.18,-117.29,...,False,False,False,False,False,False,False,False,False,False
4,46.41,13051,5813,0,51,69,1968,5,38.86,-76.60,...,False,False,False,False,False,False,False,False,False,False
5,4.81,20519,5942,0,47,65,1972,12,40.84,-73.87,...,False,False,False,False,False,False,False,False,False,False


In [18]:
# =========================================================================
# Split train/test data while preserving target class distribution.
# =========================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("=" * 50)
print("MODELLING SUMMARY")
print("=" * 50)

total_rows = len(model_df)

print(f"Total Gold Dataset Rows: {total_rows:,}")
print(f"Training Rows: {len(X_train):,}")
print(f"Testing Rows: {len(X_test):,}")
print(f"Number of Features: {X.shape[1]}")
print(f"Training Percentage: {len(X_train) / total_rows * 100:.2f}%")
print(f"Testing Percentage: {len(X_test) / total_rows * 100:.2f}%")

print("\nTraining Fraud Distribution")
print(y_train.value_counts())

print("\nTesting Fraud Distribution")
print(y_test.value_counts())

MODELLING SUMMARY
Total Gold Dataset Rows: 8,914,963
Training Rows: 7,131,970
Testing Rows: 1,782,993
Number of Features: 176
Training Percentage: 80.00%
Testing Percentage: 20.00%

Training Fraud Distribution
is_fraud
0    7121304
1      10666
Name: count, dtype: int64

Testing Fraud Distribution
is_fraud
0    1780327
1       2666
Name: count, dtype: int64


In [19]:
# =========================================================================
# Calculate class weighting to account for fraud/non-fraud imbalance.
# =========================================================================

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / max(positive_count, 1)

print("Non-fraud count:", negative_count)
print("Fraud count:", positive_count)
print("Scale pos weight:", scale_pos_weight)

Non-fraud count: 7121304
Fraud count: 10666
Scale pos weight: 667.6639789986874


In [20]:
# =========================================================================
# Train the fraud detection model.
# =========================================================================

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

print("XGBoost model training complete.")

XGBoost model training complete.


In [21]:
# =========================================================================
# Generate class predictions and probability scores.
# =========================================================================

y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:, 1]

print("Predictions generated.")

Predictions generated.


In [22]:
# =========================================================================
# Train a simple Logistic Regression baseline model.
# =========================================================================

BASELINE_SAMPLE_SIZE = 500_000

required_baseline_variables = [
    "X_train",
    "X_test",
    "y_train",
    "y_test",
]
missing_baseline_variables = [
    variable
    for variable in required_baseline_variables
    if variable not in globals()
]
if missing_baseline_variables:
    raise RuntimeError(
        "Run the data preparation, encoding, and train/test split cells before "
        "training the Logistic Regression baseline. Missing variables: "
        f"{missing_baseline_variables}"
    )

if len(X_train) > BASELINE_SAMPLE_SIZE:
    X_baseline_train, _, y_baseline_train, _ = train_test_split(
        X_train,
        y_train,
        train_size=BASELINE_SAMPLE_SIZE,
        stratify=y_train,
        random_state=42,
    )
else:
    X_baseline_train = X_train
    y_baseline_train = y_train

baseline_model = LogisticRegression(
    class_weight="balanced",
    random_state=42,
    max_iter=300,
    solver="saga",
    tol=1e-2,
    n_jobs=-1,
)

baseline_model.fit(X_baseline_train, y_baseline_train)

baseline_pred = baseline_model.predict(X_test)
baseline_proba = baseline_model.predict_proba(X_test)[:, 1]

print("Logistic Regression baseline training complete.")
print(f"Baseline training rows used: {len(X_baseline_train):,}")

c:\Install\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression baseline training complete.
Baseline training rows used: 500,000


In [23]:
# =========================================================================
# Calculate model performance metrics and compare models.
# =========================================================================

# XGBoost metrics
xgb_accuracy = accuracy_score(y_test, y_pred)
xgb_precision = precision_score(y_test, y_pred, zero_division=0)
xgb_recall = recall_score(y_test, y_pred, zero_division=0)
xgb_f1 = f1_score(y_test, y_pred, zero_division=0)
xgb_roc_auc = roc_auc_score(y_test, y_proba)
xgb_pr_auc = average_precision_score(y_test, y_proba)

# Baseline Logistic Regression metrics
baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_precision = precision_score(y_test, baseline_pred, zero_division=0)
baseline_recall = recall_score(y_test, baseline_pred, zero_division=0)
baseline_f1 = f1_score(y_test, baseline_pred, zero_division=0)
baseline_roc_auc = roc_auc_score(y_test, baseline_proba)
baseline_pr_auc = average_precision_score(y_test, baseline_proba)

# Keep the original metrics_df schema for downstream visuals (XGBoost only).
metrics_df = pd.DataFrame(
    {
        "metric": [
            "accuracy",
            "precision",
            "recall",
            "f1_score",
            "roc_auc",
            "pr_auc",
        ],
        "value": [
            xgb_accuracy,
            xgb_precision,
            xgb_recall,
            xgb_f1,
            xgb_roc_auc,
            xgb_pr_auc,
        ],
    }
)

metrics_comparison_df = pd.DataFrame(
    [
        {"model": "XGBoost", "metric": "accuracy", "value": xgb_accuracy},
        {"model": "XGBoost", "metric": "precision", "value": xgb_precision},
        {"model": "XGBoost", "metric": "recall", "value": xgb_recall},
        {"model": "XGBoost", "metric": "f1_score", "value": xgb_f1},
        {"model": "XGBoost", "metric": "roc_auc", "value": xgb_roc_auc},
        {"model": "XGBoost", "metric": "pr_auc", "value": xgb_pr_auc},
        {"model": "Logistic Regression", "metric": "accuracy", "value": baseline_accuracy},
        {"model": "Logistic Regression", "metric": "precision", "value": baseline_precision},
        {"model": "Logistic Regression", "metric": "recall", "value": baseline_recall},
        {"model": "Logistic Regression", "metric": "f1_score", "value": baseline_f1},
        {"model": "Logistic Regression", "metric": "roc_auc", "value": baseline_roc_auc},
        {"model": "Logistic Regression", "metric": "pr_auc", "value": baseline_pr_auc},
    ]
)

model_comparison_df = (
    metrics_comparison_df
    .pivot(index="metric", columns="model", values="value")
    .reset_index()
    .rename_axis(None, axis=1)
)
model_comparison_df["xgboost_minus_logistic"] = (
    model_comparison_df["XGBoost"] - model_comparison_df["Logistic Regression"]
)

print("XGBoost metrics:")
display(metrics_df)

print("Model comparison (XGBoost vs Logistic Regression):")
display(model_comparison_df.sort_values("metric"))

XGBoost metrics:


,metric,value
0,accuracy,0.949368
1,precision,0.025899
2,recall,0.897599
3,f1_score,0.050346
4,roc_auc,0.977164
5,pr_auc,0.444825


Model comparison (XGBoost vs Logistic Regression):


,metric,Logistic Regression,XGBoost,xgboost_minus_logistic
0,accuracy,0.859436,0.949368,0.089932
1,f1_score,0.006414,0.050346,0.043931
2,pr_auc,0.003861,0.444825,0.440965
3,precision,0.003241,0.025899,0.022658
4,recall,0.303451,0.897599,0.594149
5,roc_auc,0.630706,0.977164,0.346458


In [24]:
# =========================================================================
# Build a confusion matrix to show false positives and false negatives.
# =========================================================================
cm = confusion_matrix(y_test, y_pred)

confusion_matrix_df = pd.DataFrame(
    cm,
    index=["Actual Non-Fraud", "Actual Fraud"],
    columns=["Predicted Non-Fraud", "Predicted Fraud"]
)

display(confusion_matrix_df)

,Predicted Non-Fraud,Predicted Fraud
Actual Non-Fraud,1690323,90004
Actual Fraud,273,2393


In [25]:
# =========================================================================
# Produce a detailed classification report for each target class.
# =========================================================================

classification_report_df = pd.DataFrame(
    classification_report(
        y_test,
        y_pred,
        output_dict=True,
        zero_division=0
    )
).transpose()

display(classification_report_df)

,precision,recall,f1-score,support
0,0.999839,0.949445,0.973990,1.780327e+06
1,0.025899,0.897599,0.050346,2.666000e+03
accuracy,0.949368,0.949368,0.949368,9.493677e-01
macro avg,0.512869,0.923522,0.512168,1.782993e+06
weighted avg,0.998382,0.949368,0.972609,1.782993e+06


In [26]:
# =========================================================================
# Extract feature importance values from the trained XGBoost model.
# =========================================================================

feature_importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb_model.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
)

display(feature_importance_df.head(30))

,feature,importance
167,merchant_category_Tolls and Bridge Fees,0.108058
34,use_chip_Online Transaction,0.095933
164,merchant_category_Taxicabs and Limousines,0.042239
106,merchant_category_Family Clothing Stores,0.028568
94,merchant_category_Department Stores,0.028411
138,merchant_category_Money Transfer,0.027619
172,"merchant_category_Utilities - Electric, Gas, W...",0.024982
35,use_chip_Swipe Transaction,0.024661
2,mcc,0.024256
84,"merchant_category_Cable, Satellite, and Other ...",0.018530


In [27]:
# =========================================================================
# Attach predictions back to the relevant test transactions.
# =========================================================================

scored_transactions = model_df.loc[X_test.index].copy()

scored_transactions["actual_fraud"] = y_test.values
scored_transactions["predicted_fraud"] = y_pred
scored_transactions["fraud_probability"] = y_proba

# Convert fraud probabilities into simple risk bands for interpretation.
scored_transactions["fraud_risk_band"] = pd.cut(
    scored_transactions["fraud_probability"],
    bins=[0, 0.3, 0.7, 1],
    labels=["Low Risk", "Medium Risk", "High Risk"],
    include_lowest=True
)

important_scored_cols = [
    "id",
    "transaction_id",
    "amount",
    "merchant_category",
    "actual_fraud",
    "predicted_fraud",
    "fraud_probability",
    "fraud_risk_band"
]

available_scored_cols = [
    col for col in important_scored_cols if col in scored_transactions.columns]

display(scored_transactions[available_scored_cols].head(20))

,id,transaction_id,amount,merchant_category,actual_fraud,predicted_fraud,fraud_probability,fraud_risk_band
9817316,19453431,19453431.0,9.68,Eating Places and Restaurants,0,0,0.230947,Low Risk
5025049,13577387,13577387.0,1.21,Eating Places and Restaurants,0,0,0.087436,Low Risk
8804748,18210129,18210129.0,39.77,Tolls and Bridge Fees,0,0,0.008561,Low Risk
10742387,20594125,20594125.0,13.75,"Package Stores, Beer, Wine, Liquor",0,0,0.208055,Low Risk
6653150,15571088,15571088.0,66.00,Miscellaneous Food Stores,0,0,0.226838,Low Risk
12088091,22254904,22254904.0,60.00,Miscellaneous Food Stores,0,0,0.168548,Low Risk
2188102,10118591,10118591.0,45.73,Eating Places and Restaurants,0,0,0.075060,Low Risk
1331201,9076164,9076164.0,90.44,Drug Stores and Pharmacies,0,0,0.010439,Low Risk
5083472,13649023,13649023.0,-70.00,Miscellaneous Food Stores,0,0,0.014727,Low Risk
7468841,16570940,16570940.0,13.05,Service Stations,0,0,0.100468,Low Risk


In [28]:
# =========================================================================
# Create gold summary tables to provide insights into the overall fraud
# distribution, predicted risk bands, and fraud rates by merchant category.
# =========================================================================

# Summarise the overall fraud distribution.
fraud_distribution_df = gold_df[target_col].value_counts(
    dropna=False).reset_index()
fraud_distribution_df.columns = ["is_fraud", "transaction_count"]
fraud_distribution_df["percentage"] = (
    fraud_distribution_df["transaction_count"] / fraud_distribution_df["transaction_count"].sum()
) * 100

# Summarise predicted risk bands from the scored transactions.
risk_band_summary_df = scored_transactions["fraud_risk_band"].value_counts(
    dropna=False).reset_index()
risk_band_summary_df.columns = ["fraud_risk_band", "transaction_count"]

# Summarise fraud rate by merchant category where the MCC join is available.
if "merchant_category" in gold_df.columns:
    merchant_category_summary_df = (
        gold_df.groupby("merchant_category")
        .agg(
            transaction_count=(target_col, "count"),
            fraud_count=(target_col, "sum"),
            fraud_rate=(target_col, "mean")
        )
        .reset_index()
        .sort_values(by="fraud_rate", ascending=False)
    )
else:
    merchant_category_summary_df = pd.DataFrame()

print("Fraud distribution:")
display(fraud_distribution_df)

print("Risk band summary:")
display(risk_band_summary_df)

print("Merchant category summary:")
display(merchant_category_summary_df.head(20))

Fraud distribution:


,is_fraud,transaction_count,percentage
0,0,8901631,99.850454
1,1,13332,0.149546


Risk band summary:


,fraud_risk_band,transaction_count
0,Low Risk,1565864
1,Medium Risk,174758
2,High Risk,42371


Merchant category summary:


,merchant_category,transaction_count,fraud_count,fraud_rate
24,Cruise Lines,276,165,0.597826
73,Music Stores - Musical Instruments,204,76,0.372549
63,Miscellaneous Fabricated Metal Products,245,29,0.118367
22,"Computers, Computer Peripheral Equipment",1883,204,0.108338
40,Floor Covering Stores,222,23,0.103604
67,Miscellaneous Metal Fabrication,256,22,0.085938
35,Electronics Stores,4689,402,0.085733
37,Fabricated Structural Metal Products,273,22,0.080586
82,Precious Stones and Metals,3525,242,0.068652
42,"Furniture, Home Furnishings, and Equipment Stores",2600,170,0.065385


In [29]:
# =========================================================================
# Save Gold outputs for model reuse and downstream reporting.
# =========================================================================

parquet_outputs = [
    (metrics_df, gold_dir / "gold_model_metrics.parquet", False),
    (metrics_comparison_df, gold_dir / "gold_model_metrics_comparison.parquet", False),
    (model_comparison_df, gold_dir / "gold_model_comparison_summary.parquet", False),
    (confusion_matrix_df, gold_dir / "gold_confusion_matrix.parquet", True),
    (classification_report_df, gold_dir / "gold_classification_report.parquet", True),
    (feature_importance_df, gold_dir / "gold_feature_importance.parquet", False),
    (scored_transactions, gold_dir / "gold_scored_transactions.parquet", False),
    (fraud_distribution_df, gold_dir / "gold_fraud_distribution.parquet", False),
    (risk_band_summary_df, gold_dir / "gold_risk_band_summary.parquet", False),
]

for dataframe, path, include_index in parquet_outputs:
    dataframe.to_parquet(path, index=include_index)

if not merchant_category_summary_df.empty:
    merchant_category_summary_df.to_parquet(
        gold_dir / "gold_merchant_category_summary.parquet",
        index=False,
    )

joblib.dump(xgb_model, gold_dir / "xgboost_fraud_model.pkl")
joblib.dump(baseline_model, gold_dir / "logistic_regression_baseline.pkl")

print("Gold outputs saved successfully.")
print("Saved to:", gold_dir)

Gold outputs saved successfully.
Saved to: C:\Users\9370892\OneDrive - Lloyds Banking Group\Apprenticeship - Data Science\Data Science Pro Practice\Assessment\Data\gold


In [30]:
# =========================================================================
# Final summary of Gold layer outputs and model details.
# =========================================================================

print("Gold Layer Complete")
print("-------------------")
print("Primary model: XGBoost Classifier")
print("Baseline model: Logistic Regression")
print("Target column:", target_col)
print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])
print("Number of model features:", X.shape[1])

print("XGBoost metrics (used for downstream visuals):")
display(metrics_df)

print("Model comparison summary (XGBoost vs Logistic Regression):")
display(model_comparison_df.sort_values("metric"))

print("Top 10 most important XGBoost features:")
display(feature_importance_df.head(10))

print("Gold output files created:")
for file in sorted(gold_dir.glob("gold_*.parquet")):
    print(file.name)

print("Model files:")
print("xgboost_fraud_model.pkl")
print("logistic_regression_baseline.pkl")

Gold Layer Complete
-------------------
Primary model: XGBoost Classifier
Baseline model: Logistic Regression
Target column: is_fraud
Training records: 7131970
Testing records: 1782993
Number of model features: 176
XGBoost metrics (used for downstream visuals):


,metric,value
0,accuracy,0.949368
1,precision,0.025899
2,recall,0.897599
3,f1_score,0.050346
4,roc_auc,0.977164
5,pr_auc,0.444825


Model comparison summary (XGBoost vs Logistic Regression):


,metric,Logistic Regression,XGBoost,xgboost_minus_logistic
0,accuracy,0.859436,0.949368,0.089932
1,f1_score,0.006414,0.050346,0.043931
2,pr_auc,0.003861,0.444825,0.440965
3,precision,0.003241,0.025899,0.022658
4,recall,0.303451,0.897599,0.594149
5,roc_auc,0.630706,0.977164,0.346458


Top 10 most important XGBoost features:


,feature,importance
167,merchant_category_Tolls and Bridge Fees,0.108058
34,use_chip_Online Transaction,0.095933
164,merchant_category_Taxicabs and Limousines,0.042239
106,merchant_category_Family Clothing Stores,0.028568
94,merchant_category_Department Stores,0.028411
138,merchant_category_Money Transfer,0.027619
172,"merchant_category_Utilities - Electric, Gas, W...",0.024982
35,use_chip_Swipe Transaction,0.024661
2,mcc,0.024256
84,"merchant_category_Cable, Satellite, and Other ...",0.018530


Gold output files created:
gold_classification_report.parquet
gold_confusion_matrix.parquet
gold_feature_importance.parquet
gold_fraud_distribution.parquet
gold_merchant_category_summary.parquet
gold_model_comparison_summary.parquet
gold_model_metrics.parquet
gold_model_metrics_comparison.parquet
gold_risk_band_summary.parquet
gold_scored_transactions.parquet
Model files:
xgboost_fraud_model.pkl
logistic_regression_baseline.pkl


## Report Summary Text

You can include this wording in your report:

> The Gold layer used the cleaned Silver Parquet datasets to create a modelling dataset for fraud detection. Transaction records were enriched with user, card, fraud label and MCC merchant category data. The MCC join added merchant category context, allowing the model to consider the type of merchant associated with each transaction. Additional features were engineered, including time-based transaction variables, debt-to-income ratio, high-value transaction flags, credit risk indicators and card-related risk flags.
>
> An XGBoost classification model was trained using the `is_fraud` label as the target variable. The dataset was split into training and test sets using stratification to preserve the fraud/non-fraud balance. Class imbalance was handled using `scale_pos_weight`, which increased the model's focus on the minority fraud class.
>
> Model performance was evaluated using accuracy, precision, recall, F1-score, ROC-AUC and PR-AUC. Recall was considered especially important in the fraud detection context because it measures how many actual fraud cases were correctly identified. The Gold layer outputs included model metrics, a confusion matrix, classification report, feature importance, scored transactions, risk band summaries, merchant category summaries and the saved XGBoost model.
